## tl;dr

V11.17: 0/6 exploratory survivors; certified Alpha remains0.
本笔记复核已审计结果，不重新搜索、不拟合模型、不打开封存数据。


## Context & Methods

Companion to the bilingual V11.17 reports and Issue184.
### Key Assumptions

2023–2024 are reused development history; CNY3m;82/164bps.
Statistical peers are not actual supply chains. All50 account outcomes remain visible.
Install project research extras to obtain DuckDB; the code cells need only standard Python and DuckDB.
Raw-feature and graph checks are implemented in `scripts/peer_source_audit.sql` and `scripts/audit_peer_information.py`.


## Data

### 1. Read bounded audit evidence


In [1]:
import json
from pathlib import Path
import hashlib
import duckdb

repo = Path.cwd()
if not (repo / 'docs/V11_17_RESULT.summary.json').exists():
    repo = repo.parent
operation = repo / 'artifacts/peer-information/epoch-001'
summary = json.loads((repo / 'docs/V11_17_RESULT.summary.json').read_text(encoding='utf-8'))
audit = json.loads((operation / 'INDEPENDENT_AUDIT.json').read_text(encoding='utf-8'))
assert audit['pass'] and summary['independent_audit_pass']
assert hashlib.sha256((operation / 'RESULT.json').read_bytes()).hexdigest() == summary['source_result_sha256']
assert len(summary['rows']) == 50 and summary['raw_trial_lower_bound'] == 3556
print({'accounts': 50, 'source_rows_checked': summary['input_rows'], 'debt': 3556,
       'graph_checks': summary['graph_checks'], 'coverage': summary['coverage']})


{'accounts': 50, 'source_rows_checked': 3440702, 'debt': 3556, 'graph_checks': {'selected_edges': 92060, 'top10_receivers_sampled': 40}, 'coverage': {'2023': {'mean_common_ratio': 0.927074035655502}, '2024': {'mean_common_ratio': 0.9527922921262866}}}


### 2. Verify the saved source-query identity

This checks the query bound by the completed audit; it does not claim to re-execute the heavy raw-source join here.


In [2]:
source_sql = (repo / 'scripts/peer_source_audit.sql').read_text(encoding='utf-8')
assert source_sql == audit['source_query']
print({'raw_source_query_bound': True, 'query_lines': len(source_sql.splitlines())})


{'raw_source_query_bound': True, 'query_lines': 41}


## Results

### 3. Recompute all50 account aggregates with independent SQL


In [3]:
import math
query = (repo / 'scripts/lead_challenge_audit.sql').read_text(encoding='utf-8')
query = query.replace('__ACCOUNT_GLOB__', (operation / 'accounts/*.jsonl').as_posix())
with duckdb.connect() as connection:
    cursor = connection.execute(query)
    columns = [column[0] for column in cursor.description]
    recomputed = {row[0]: dict(zip(columns, row)) for row in cursor.fetchall()}
assert len(recomputed) == 50
for row in summary['rows']:
    actual = recomputed[row['account_key']]
    for field in ('net_return', 'final_nav', 'cost_cny', 'max_drawdown'):
        assert math.isclose(actual[field], row[field], rel_tol=1e-10, abs_tol=1e-6)
print({'independently_recomputed_accounts': len(recomputed), 'aggregate_reconciliation': True})


{'independently_recomputed_accounts': 50, 'aggregate_reconciliation': True}


### 4. Inspect every primary identity, without winner-only filtering


In [4]:
for row in summary['rows']:
    if row['policy'] == 'peer':
        print({key: row[key] for key in ('identity', 'roundtrip_bps', 'return2023', 'return2024', 'net_return', 'sharpe', 'max_drawdown')})
print({'exploratory_survivors': summary['screen_survived'], 'validated_alpha': summary['validated_alpha']})


{'identity': 'flow-high', 'roundtrip_bps': 164, 'return2023': -0.32796278243665256, 'return2024': -0.3933906180673721, 'net_return': -0.5923359188181749, 'sharpe': -1.5416187311523624, 'max_drawdown': -0.6866217485142498}
{'identity': 'flow-high', 'roundtrip_bps': 82, 'return2023': -0.261297582071932, 'return2024': -0.3306147839095297, 'net_return': -0.5055235223486688, 'sharpe': -1.191213729591828, 'max_drawdown': -0.6333129907629276}
{'identity': 'flow-low', 'roundtrip_bps': 164, 'return2023': -0.1383538096865602, 'return2024': -0.008556017175004427, 'net_return': -0.14572606928965692, 'sharpe': -0.31202415810103645, 'max_drawdown': -0.3895262961012331}
{'identity': 'flow-low', 'roundtrip_bps': 82, 'return2023': -0.05741270907062712, 'return2024': 0.08649736258503582, 'net_return': 0.024118605600936593, 'sharpe': 0.16119251921498254, 'max_drawdown': -0.29209121401665994}
{'identity': 'flow-middle', 'roundtrip_bps': 164, 'return2023': -0.1820346912480303, 'return2024': -0.153448342029

## Takeaways

0/6 pass the full exploratory screen, not Alpha Court.
All native trial, graph and account evidence remains available locally. No final-test reopening or trading is authorized.
Validation: every plain-Python code cell executed in order by `scripts/build_peer_notebook.py`.
Jupyter frontend/kernel QA was not run: nbformat,nbclient and ipykernel are not installed.
Optional frontend check after installing Jupyter: `python -m jupyter nbconvert --execute --to notebook --inplace notebooks/V11_17_AUDIT.ipynb`.
